# Becoming a Backprop Ninja（详细中文注释 + 手推梯度全解版）

Karpathy《building makemore **Part 4**: Becoming a Backprop Ninja》跟练笔记本。

**这一讲在干嘛**:前几讲我们都靠 `loss.backward()` 让 PyTorch 自动求梯度。这一讲把这个「黑盒」彻底打开——
**从 `loss` 出发,一步步手推每个中间变量的梯度**,一直反推到参数 `C, W1, b1, W2, b2, bngain, bnbias`,
然后用 `cmp()` 跟 PyTorch 的自动求导**逐个对拍**,`exact: True` 就说明你推对了。

> **本笔记本已按你的要求:把所有空练习格都填好了正确答案,并且每一行都写了「这个梯度是怎么从微积分推出来的」中文注释。**
> 代码 cell 0~7(前向传播和准备)保持你原来的写法**一字未改**,只加了注释。

---

### 先给你吃颗定心丸:反向传播用不到高深微积分

很多人一看「手推梯度」就怕,其实**backprop 只用到一小撮最基础的求导 + 一条链式法则**,下面这节速成把它们全讲了。看完你就会发现:所谓「手推梯度」= 查下面这张小表 + 注意张量形状,如此而已。

## 📐 反向传播微积分速成（从零讲起，看完就够用）

### 1. 导数是什么
`导数 = 变化率`。`d(loss)/dx` 读作「x 动一点点,loss 会跟着动多少倍」。反向传播要算的,就是 loss 对**每一个**中间量的导数(梯度)。记号约定:代码里 `dx` 就代表 `d(loss)/dx`,且 **`dx` 的形状永远和 `x` 一样**。

### 2. 链式法则（整个反向传播的唯一核心）
如果 loss 是通过中间量 `y` 才依赖 `x` 的,那么:

$$\frac{d\,loss}{dx} = \frac{d\,loss}{dy}\cdot\frac{dy}{dx}$$

翻译成人话:**`dx = (上游传来的梯度 dy) × (y 对 x 的局部导数)`**。
反向传播就是从 loss 开始,一层层把「上游梯度」乘上「本地局部导数」,往回传。就这一条,反复用。

### 3. 常见函数的局部导数（一张小表，够用了）
| 前向 | 局部导数 `dy/dx` | 反向写法(`dy` 是上游梯度) |
|---|---|---|
| `y = a*x`（a 是常数） | `a` | `dx = a * dy` |
| `y = x**k`（幂） | `k*x**(k-1)` | `dx = k*x**(k-1) * dy` |
| `y = exp(x)` | `exp(x) = y` | `dx = y * dy` |
| `y = log(x)`（自然对数） | `1/x` | `dx = (1/x) * dy` |
| `y = tanh(x)` | `1 - tanh(x)**2 = 1 - y**2` | `dx = (1 - y**2) * dy` |
| `y = a + x` | `1` | `dx = dy`（加法:梯度原样传过去） |
| `y = a * x`（逐元素乘） | `a` | `dx = a * dy` |

> 幂那一行覆盖了本讲的 `**2`(平方)、`**-1`(倒数)、`**-0.5`(开方求逆) 等所有情况——套公式即可。

### 4. 两个「张量特有」的规则（新手最容易错的地方）

**(a) 求和 sum ↔ 广播 broadcast 是一对互逆操作**
- 前向 `y = x.sum(0)`(把 32 行加成 1 行):反向时每个 x 元素都平等贡献,所以 `dx = 广播(dy)`,即把 `dy` 复制回 (32, ...) 形状。
- 前向如果发生了**广播**(比如 `(1,64)` 的东西被自动复制成 `(32,64)` 参与运算):反向时要把梯度 **`sum` 回原来的小形状**。

一句口诀:**前向 sum 了 → 反向 broadcast;前向 broadcast 了 → 反向 sum。最后 `dx` 形状必须和 `x` 一样。**

**(b) 矩阵乘法 `Y = X @ W`** 的梯度(直接背结论,用形状凑就不会错):

$$dX = dY @ W^{T} \qquad dW = X^{T} @ dY$$

记忆法:`X` 是 `(a,b)`、`W` 是 `(b,c)`、`Y` 是 `(a,c)`。想让 `dX` 得到 `(a,b)`,只能 `dY(a,c) @ W.T(c,b)`;想让 `dW` 得到 `(b,c)`,只能 `X.T(b,a) @ dY(a,c)`。**形状对上了,就对了。** 偏置 `Y = X@W + b` 时 `db = dY.sum(0)`(因为 b 被广播到每一行)。

### 5. 一个漂亮结论（本讲高潮）
softmax + 交叉熵合起来求导,结果极其干净:

$$d\,logits = softmax(logits) - \text{onehot}(y),\ \text{再除以 } n$$

即「预测概率 - 真实答案」。这就是为什么框架里 `cross_entropy` 又稳又快——梯度不用绕一大圈,一步到位。(下面 Exercise 2 会验证它。)

---

**准备好了。下面每一步手推梯度,都只是在用上面这张表 + 链式法则 + 形状规则。遇到不确定的,回来查这一节。**

## 0. 导入库

In [ ]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt # for making figures
%matplotlib inline

## 1. 读数据

In [ ]:
words = open('names.txt', 'r').read().splitlines()   # 读入所有名字(约 32033 个)
print(len(words))                                    # 名字总数
print(max(len(w) for w in words))                    # 最长名字的长度
print(words[:8])                                     # 看前 8 个

## 2. 建词表(字符↔编号)

In [ ]:
chars = sorted(list(set(''.join(words))))            # 所有字符去重排序(a..z)
stoi = {s:i+1 for i,s in enumerate(chars)}           # 字符->编号: a=1..z=26
stoi['.'] = 0                                        # '.' 起止符,编号 0
itos = {i:s for s,i in stoi.items()}                 # 编号->字符
vocab_size = len(itos)                               # 词表大小 = 27
print(itos)
print(vocab_size)

## 3. 构造数据集 + 切分 训练/验证/测试

In [ ]:
block_size = 3                                       # 用前 3 个字符预测下一个

def build_dataset(words):                            # 名字列表 -> (X, Y)
  X, Y = [], []

  for w in words:
    context = [0] * block_size                       # 上下文初始化 [0,0,0]
    for ch in w + '.':                               # 遍历字符,末尾补 '.'
      ix = stoi[ch]                                  # 当前字符编号
      X.append(context)                              # 上下文 ->
      Y.append(ix)                                   #        目标
      context = context[1:] + [ix]                   # 窗口右移一格

  X = torch.tensor(X)                                # (样本数, 3)
  Y = torch.tensor(Y)                                # (样本数,)
  print(X.shape, Y.shape)
  return X, Y

import random
random.seed(42)                                      # 固定种子,可复现
random.shuffle(words)                                # 打乱名字
n1 = int(0.8*len(words))                             # 80% 分界
n2 = int(0.9*len(words))                             # 90% 分界

Xtr,  Ytr  = build_dataset(words[:n1])               # 训练集 80%
Xdev, Ydev = build_dataset(words[n1:n2])             # 验证集 10%
Xte,  Yte  = build_dataset(words[n2:])               # 测试集 10%

## 4. 对拍工具 `cmp`(本讲的裁判)

**为什么需要它**:我们手推的梯度对不对,得有个标准答案对照。`t.grad` 是 PyTorch 自动求导算出的「正确答案」,`dt` 是我们手推的。`cmp` 就是把两者比一比:
- `exact: True` = 完全逐位相等(最理想);
- `approximate: True` = 在浮点误差内相等(用了不同算法时会这样,也算对);
- `maxdiff` = 两者最大差多少。

In [ ]:
def cmp(s,dt,t):                                     # s=名字, dt=手推梯度, t=前向张量(t.grad 是标准答案)
    ex=torch.all(dt==t.grad).item()                  # 是否逐位完全相等
    app=torch.allclose(dt,t.grad)                    # 是否在浮点误差内相等
    maxdiff=(dt-t.grad).abs().max().item()           # 最大差值
    print(f'{s:15s} | exact: {str(ex):5s} | approximate: {str(app):5s} | maxdiff: {maxdiff}')

## 5. 初始化参数

和 Part 3 一样的 MLP+BatchNorm 结构;这里 BN 的 `bngain/bnbias` 特意用了随机值(不是标准 1/0),是为了让手推梯度更有代表性、别碰巧算对。

In [ ]:
n_emb=10                                             # 每个字符嵌入 10 维
n_hidden=64                                          # 隐藏层宽度
g=torch.Generator().manual_seed(2147483647)          # 固定种子
C=torch.randn((vocab_size,n_emb),generator=g)        # 嵌入表 (27,10)
W1 = torch.randn((n_emb * block_size, n_hidden), generator=g) * (5/3)/((n_emb * block_size)**0.5)  # 第一层权重 (30,64),tanh Kaiming 初始化
b1=torch.randn(n_hidden,generator=g)*0.1             # 第一层偏置(本讲保留了 b1,可对照观察它的梯度)
W2=torch.randn((n_hidden,vocab_size),generator=g)*0.1  # 输出层权重 (64,27)
b2 = torch.randn(vocab_size,generator=g) * 0.1       # 输出层偏置 (27,)
bngain = torch.randn((1, n_hidden))*0.1 + 1.0        # BN 缩放 γ,随机在 1 附近
bnbias = torch.randn((1, n_hidden))*0.1              # BN 平移 β,随机在 0 附近
parameters=[C,W1,b1,W2,b2,bngain,bnbias]             # 所有可训练参数
print(sum(p.nelement() for p in parameters))         # 参数总量
for p in parameters: p.requires_grad=True            # 开启梯度追踪

## 6. 取一个 minibatch

In [ ]:
batch_size = 32                                      # 小批量大小
n=batch_size                                         # 后面公式频繁用到 n,起个短名
ix=torch.randint(0,Xtr.shape[0],(batch_size,),generator=g)  # 随机抽 32 个下标
Xb, Yb = Xtr[ix], Ytr[ix]                            # 这一批的输入/目标

## 7. 前向传播（故意拆成很多小步）

**为什么拆这么碎**:正常写一行 `F.cross_entropy` 就完事了,这里故意把 softmax、交叉熵、BatchNorm 全**手动摊开成一个个中间变量**(`logit_maxes`、`counts`、`bndiff`...)。因为我们要对**每一个**中间变量手推梯度,拆得越碎,每一步的求导就越简单。

末尾 `retain_grad()` 是**关键**:PyTorch 默认只保留叶子节点(参数)的梯度,中间变量的 `.grad` 反向后会丢。`retain_grad()` 把它们全留住,这样 `cmp` 才能拿它们当标准答案。

In [ ]:
emb=C[Xb]                                            # (32,3,10) 查嵌入
embcat=emb.view(emb.shape[0],-1)                     # (32,30) 拼平
hprebn=embcat@W1+b1                                  # (32,64) 第一层线性(BN 前)
# --- BatchNorm 手动展开 ---
bnmeani=1/n*hprebn.sum(0,keepdim=True)               # (1,64) 每个神经元这批的均值
bndiff=hprebn-bnmeani                                # (32,64) 去均值
bndiff2=bndiff**2                                    # (32,64) 平方
bnvar=1/(n-1)*(bndiff2).sum(0,keepdim=True)          # (1,64) 方差(Bessel 修正:除 n-1)
bnvar_inv = (bnvar + 1e-5)**-0.5                     # (1,64) 1/标准差
bnraw = bndiff * bnvar_inv                           # (32,64) 归一化
hpreact = bngain * bnraw + bnbias                    # (32,64) γ 缩放 + β 平移
# --- 激活 + 输出层 ---
h = torch.tanh(hpreact)                              # (32,64) tanh
logits = h @ W2 + b2                                 # (32,27) 输出得分
# --- softmax + 交叉熵 手动展开 ---
logit_maxes = logits.max(1, keepdim=True).values     # (32,1) 每行最大值(为数值稳定)
norm_logits = logits - logit_maxes                   # (32,27) 减最大值,防 exp 溢出
counts = norm_logits.exp()                           # (32,27) exp
counts_sum = counts.sum(1, keepdims=True)            # (32,1) 每行求和
counts_sum_inv = counts_sum**-1                      # (32,1) 倒数
probs = counts * counts_sum_inv                      # (32,27) 归一化成概率
logprobs = probs.log()                               # (32,27) 取对数
loss = -logprobs[range(n), Yb].mean()                # 取正确目标的 logprob,取负求均值 = 交叉熵
# ---
for p in parameters:
  p.grad = None
for t in [logprobs, probs, counts, counts_sum, counts_sum_inv,
          norm_logits, logit_maxes, logits, h, hpreact, bnraw,
         bnvar_inv, bnvar, bndiff2, bndiff, hprebn, bnmeani,
         embcat, emb]:
  t.retain_grad()                                    # 保留所有中间变量的梯度,供 cmp 对拍
loss.backward()                                      # PyTorch 自动求导,得到「标准答案」t.grad
loss

---

# Exercise 1：逐个变量手推梯度（从 loss 一路反推到 C, W1, ...）

下面按前向的**逆序**,一个变量一个变量往回推。每个 `dxxx` 都用上面速成里的规则算出,并立刻 `cmp` 验证。我把它分成 4 段(交叉熵区 → 输出层 → BatchNorm → 第一层/嵌入),每段末尾都对拍。**全部 `exact: True` 就说明整条链推对了。**

### 1-A. 交叉熵 / softmax 区：`dlogprobs → dlogits`

关键几步的推导:
- `loss = -1/n · Σ logprobs[i, Yb[i]]`:只有被选中的那 32 个元素参与,所以 `dlogprobs` 只在这些位置是 `-1/n`,其余为 0。
- `logprobs = log(probs)` → 局部导数 `1/probs`(查表 log)。
- `probs = counts * counts_sum_inv`:`counts_sum_inv` 是 `(32,1)` **被广播**到 27 列 → 对它求梯度要 **`sum` 回 (32,1)**(形状规则!);对 `counts` 则是逐元素乘。
- `counts_sum_inv = counts_sum**-1` → 局部导数 `-counts_sum**-2`(查表幂)。
- `counts = exp(norm_logits)` → 局部导数就是 `counts` 本身(查表 exp)。
- `norm_logits = logits - logit_maxes`:减法梯度原样传;`logit_maxes` 那支梯度理论上≈0(减最大值只为数值稳定),但为 `exact` 也照推。

In [ ]:
# loss = -mean(被选中的 logprobs) -> 只有 (i, Yb[i]) 位置有梯度 -1/n
dlogprobs = torch.zeros_like(logprobs)               # 先全 0,形状同 logprobs
dlogprobs[range(n), Yb] = -1.0/n                     # 被选中的位置 = -1/n
cmp('logprobs', dlogprobs, logprobs)

# logprobs = log(probs) -> 局部导数 1/probs (查表 log)
dprobs = (1.0/probs) * dlogprobs
cmp('probs', dprobs, probs)

# probs = counts * counts_sum_inv;counts_sum_inv 被广播(1列->27列),反向要 sum 回来
dcounts_sum_inv = (counts * dprobs).sum(1, keepdim=True)   # 对广播维求和 -> (32,1)
cmp('counts_sum_inv', dcounts_sum_inv, counts_sum_inv)

dcounts = counts_sum_inv * dprobs                    # counts 的第 1 支梯度(它还喂了 counts_sum,稍后补第 2 支)

# counts_sum_inv = counts_sum**-1 -> 局部导数 -counts_sum**-2 (查表幂)
dcounts_sum = (-counts_sum**-2) * dcounts_sum_inv
cmp('counts_sum', dcounts_sum, counts_sum)

# counts_sum = counts.sum(1):前向 sum 了 -> 反向广播回去,给 counts 补上第 2 支梯度
dcounts += torch.ones_like(counts) * dcounts_sum
cmp('counts', dcounts, counts)

# counts = exp(norm_logits) -> 局部导数 = counts 本身 (查表 exp)
dnorm_logits = counts * dcounts
cmp('norm_logits', dnorm_logits, norm_logits)

# norm_logits = logits - logit_maxes:logits 支梯度原样传;logit_maxes 支要 sum 且取负
dlogits = dnorm_logits.clone()                       # logits 第 1 支
dlogit_maxes = (-dnorm_logits).sum(1, keepdim=True)  # logit_maxes 支(理论≈0)
cmp('logit_maxes', dlogit_maxes, logit_maxes)

# logit_maxes = logits.max(1):梯度只回流到每行最大值所在位置
dlogits += F.one_hot(logits.max(1).indices, num_classes=logits.shape[1]) * dlogit_maxes
cmp('logits', dlogits, logits)

### 1-B. 输出层 + tanh：`dh, dW2, db2, dhpreact`

- `logits = h @ W2 + b2`:套矩阵乘公式 `dh = dlogits @ W2.T`、`dW2 = h.T @ dlogits`、`db2 = dlogits.sum(0)`(b2 被广播到每一行,反向 sum)。
- `h = tanh(hpreact)`:局部导数 `1 - h**2`(查表 tanh)。

In [ ]:
# logits = h @ W2 + b2 -> 矩阵乘公式(用形状凑)
dh = dlogits @ W2.T                                  # (32,27)@(27,64)=(32,64)
dW2 = h.T @ dlogits                                  # (64,32)@(32,27)=(64,27)
db2 = dlogits.sum(0)                                 # b2 被广播到每行,反向 sum -> (27,)
cmp('h', dh, h); cmp('W2', dW2, W2); cmp('b2', db2, b2)

# h = tanh(hpreact) -> 局部导数 1 - h**2 (查表 tanh)
dhpreact = (1.0 - h**2) * dh
cmp('hpreact', dhpreact, hpreact)

### 1-C. BatchNorm 区：`dbngain, dbnbias, dbnraw → ... → dhprebn`

这一段最长,但每一步还是那几条规则。注意所有 `(1,64)` 的量(γ、β、bnvar_inv、bnvar、bnmeani)都是**被广播**到 `(32,64)` 的,所以对它们求梯度都要 **`sum(0)`** 回 `(1,64)`。

In [ ]:
# hpreact = bngain * bnraw + bnbias;bngain/bnbias 是(1,64)被广播,反向 sum(0)
dbngain = (bnraw * dhpreact).sum(0, keepdim=True)    # (1,64)
dbnraw = bngain * dhpreact                           # (32,64)
dbnbias = dhpreact.sum(0, keepdim=True)              # (1,64)
cmp('bngain', dbngain, bngain); cmp('bnraw', dbnraw, bnraw); cmp('bnbias', dbnbias, bnbias)

# bnraw = bndiff * bnvar_inv;bnvar_inv(1,64)被广播,反向 sum(0)
dbndiff = bnvar_inv * dbnraw                         # bndiff 第 1 支
dbnvar_inv = (bndiff * dbnraw).sum(0, keepdim=True)  # (1,64)
cmp('bnvar_inv', dbnvar_inv, bnvar_inv)

# bnvar_inv = (bnvar + 1e-5)**-0.5 -> 局部导数 -0.5*(bnvar+1e-5)**-1.5 (查表幂)
dbnvar = (-0.5 * (bnvar + 1e-5)**-1.5) * dbnvar_inv
cmp('bnvar', dbnvar, bnvar)

# bnvar = 1/(n-1) * bndiff2.sum(0):前向 sum 了 -> 反向广播,并带上 1/(n-1) 常数
dbndiff2 = (1.0/(n-1)) * torch.ones_like(bndiff2) * dbnvar
cmp('bndiff2', dbndiff2, bndiff2)

# bndiff2 = bndiff**2 -> 局部导数 2*bndiff (查表幂);给 bndiff 补第 2 支
dbndiff += (2*bndiff) * dbndiff2
cmp('bndiff', dbndiff, bndiff)

# bndiff = hprebn - bnmeani:hprebn 支原样传;bnmeani 支 sum 且取负
dhprebn = dbndiff.clone()                            # hprebn 第 1 支
dbnmeani = (-dbndiff).sum(0, keepdim=True)           # (1,64)
cmp('bnmeani', dbnmeani, bnmeani)

# bnmeani = 1/n * hprebn.sum(0):前向 sum -> 反向广播,带 1/n;给 hprebn 补第 2 支
dhprebn += (1.0/n) * torch.ones_like(hprebn) * dbnmeani
cmp('hprebn', dhprebn, hprebn)

### 1-D. 第一层 + 嵌入：`dembcat, dW1, db1 → demb → dC`

- `hprebn = embcat @ W1 + b1`:又是矩阵乘公式。
- `embcat = emb.view(...)`:`view` 只是**改形状不改数值**,梯度只需 `view` 回原形状。
- `emb = C[Xb]`:这是**按索引取行**。反向就是它的逆——**把梯度按同样的索引累加回 C**(同一个字符可能被多次取用,所以要 `+=` 累加,不能覆盖)。

In [ ]:
# hprebn = embcat @ W1 + b1 -> 矩阵乘公式
dembcat = dhprebn @ W1.T                             # (32,64)@(64,30)=(32,30)
dW1 = embcat.T @ dhprebn                             # (30,32)@(32,64)=(30,64)
db1 = dhprebn.sum(0)                                 # (64,)
cmp('embcat', dembcat, embcat); cmp('W1', dW1, W1); cmp('b1', db1, b1)

# embcat = emb.view(32,30):只是变形,梯度 view 回原形状 (32,3,10)
demb = dembcat.view(emb.shape)
cmp('emb', demb, emb)

# emb = C[Xb]:按索引取行的逆 = 按索引把梯度累加回 C
dC = torch.zeros_like(C)
for k in range(Xb.shape[0]):                         # 遍历 batch 里 32 个样本
  for j in range(Xb.shape[1]):                       # 每个样本 3 个字符
    ix = Xb[k,j]                                     # 该位置用的字符编号
    dC[ix] += demb[k,j]                              # 把梯度累加到 C 的第 ix 行
cmp('C', dC, C)

---

# Exercise 2：`dlogits` 一步到位（softmax + 交叉熵合并求导）

Exercise 1 里我们绕了 9 步(`dlogprobs→dprobs→...→dlogits`)才推到 `dlogits`。但如果把 softmax 和交叉熵**合起来一次性求导**,数学上会化简成一个极其干净的结果(速成第 5 条):

$$d\,logits = softmax(logits) - \text{onehot}(y),\ \text{再} \div n$$

即 **「预测概率 − 真实答案」**。直觉:某个字符预测概率越比真实值高,梯度就越大、越要被压下去。这就是框架 `F.cross_entropy` 内部又快又稳的原因。下面验证它和 Exercise 1 的 `dlogits` 一致(浮点级 `approximate: True`)。

In [ ]:
dlogits_fast = F.softmax(logits, 1)                  # 预测概率 softmax(logits)
dlogits_fast[range(n), Yb] -= 1                      # 减去真实答案的 one-hot(只在正确类别处 -1)
dlogits_fast /= n                                    # 因为 loss 取了 mean,要除以 n
cmp('logits', dlogits_fast, logits)                  # 和 Exercise 1 对拍:approximate True 即正确

---

# Exercise 3：`dhprebn` 一步到位（把整个 BatchNorm 合并求导）

Exercise 1 里 BatchNorm 也绕了 8 步。如果把整个 BN(减均值、除标准差)当一个整体求导、把中间量代入化简,会得到下面这个著名的紧凑公式。它一步就从 `dhpreact`(准确说是 `dbnraw`,这里用等价的 `dhpreact` 路径)直接得到 `dhprebn`:

```
dhprebn = bngain*bnvar_inv/n * ( n*dhpreact
                                 - dhpreact.sum(0)
                                 - n/(n-1)*bnraw*(dhpreact*bnraw).sum(0) )
```

**为什么这么绕**:因为 BN 里每个样本的输出都依赖「整批的均值和方差」,所以某个 `hprebn[i]` 的改动会通过均值/方差影响到**同一批所有样本**——这就是公式里那两个 `.sum(0)` 的来历(把整批的相互影响收集起来)。`n/(n-1)` 来自方差用了 Bessel 修正(除 `n-1`)。生产里 `nn.BatchNorm1d` 内部用的就是这类化简公式,不会真去存一堆中间变量。

In [ ]:
# 一次性穿过 BatchNorm(dhpreact 已在 Exercise 1-B 算好)
dhprebn_fast = bngain*bnvar_inv/n * (n*dhpreact - dhpreact.sum(0) - n/(n-1)*bnraw*(dhpreact*bnraw).sum(0))
cmp('hprebn', dhprebn_fast, hprebn)                  # 和 Exercise 1 对拍:approximate True 即正确

---

# Exercise 4：用「手写反向传播」跑一段真实训练（不再用 loss.backward()）

把上面推导的紧凑梯度(Exercise 2 的 `dlogits` + Exercise 3 的 `dhprebn`)串起来,**完全手动**算出所有参数梯度并更新,跑一小段训练,证明我们的手推 backprop 真能训练网络。

> ✅ **安全说明(别让电脑卡死)**:这里 `max_steps` 只设了 **2000 步**(几秒钟),纯粹为了演示手写 backprop 能让 loss 下降。想练得更久,把 `max_steps` 调大即可——每步都是纯张量运算,不会卡机,只是变慢。

In [ ]:
# 重新初始化一套参数(和前面独立)
g2 = torch.Generator().manual_seed(2147483647)
C  = torch.randn((vocab_size, n_emb), generator=g2)
W1 = torch.randn((n_emb*block_size, n_hidden), generator=g2) * (5/3)/((n_emb*block_size)**0.5)
b1 = torch.randn(n_hidden, generator=g2) * 0.1
W2 = torch.randn((n_hidden, vocab_size), generator=g2) * 0.1
b2 = torch.randn(vocab_size, generator=g2) * 0.1
bngain = torch.randn((1, n_hidden), generator=g2)*0.1 + 1.0
bnbias = torch.randn((1, n_hidden), generator=g2)*0.1
parameters = [C, W1, b1, W2, b2, bngain, bnbias]
for p in parameters: p.requires_grad = True

max_steps = 2000                                     # 只跑 2000 步,安全快速
batch_size = 32; n = batch_size
lossi = []
for i in range(max_steps):
    # --- minibatch ---
    ix = torch.randint(0, Xtr.shape[0], (batch_size,), generator=g2)
    Xb, Yb = Xtr[ix], Ytr[ix]
    # --- 前向(和 cell 7 相同,略去中间 retain) ---
    emb = C[Xb]; embcat = emb.view(emb.shape[0], -1)
    hprebn = embcat @ W1 + b1
    bnmean = hprebn.mean(0, keepdim=True)
    bnvar = hprebn.var(0, keepdim=True, unbiased=True)
    bnvar_inv = (bnvar + 1e-5)**-0.5
    bnraw = (hprebn - bnmean) * bnvar_inv
    hpreact = bngain * bnraw + bnbias
    h = torch.tanh(hpreact)
    logits = h @ W2 + b2
    loss = F.cross_entropy(logits, Yb)
    # --- 手写反向传播(用 Exercise 2/3 的紧凑公式) ---
    for p in parameters: p.grad = None
    # dlogits: softmax - onehot, /n
    dlogits = F.softmax(logits, 1); dlogits[range(n), Yb] -= 1; dlogits /= n
    # 输出层
    dh = dlogits @ W2.T
    dW2 = h.T @ dlogits
    db2 = dlogits.sum(0)
    # tanh
    dhpreact = (1.0 - h**2) * dh
    # BN 参数
    dbngain = (bnraw * dhpreact).sum(0, keepdim=True)
    dbnbias = dhpreact.sum(0, keepdim=True)
    # BN 紧凑公式 -> dhprebn
    dhprebn = bngain*bnvar_inv/n * (n*dhpreact - dhpreact.sum(0) - n/(n-1)*bnraw*(dhpreact*bnraw).sum(0))
    # 第一层
    dembcat = dhprebn @ W1.T
    dW1 = embcat.T @ dhprebn
    db1 = dhprebn.sum(0)
    # 嵌入
    demb = dembcat.view(emb.shape)
    dC = torch.zeros_like(C)
    for k in range(Xb.shape[0]):
        for j in range(Xb.shape[1]):
            dC[Xb[k,j]] += demb[k,j]
    grads = [dC, dW1, db1, dW2, db2, dbngain, dbnbias]
    # --- 手动更新 ---
    lr = 0.1 if i < 1000 else 0.01
    for p, grad in zip(parameters, grads):
        p.data += -lr * grad
    lossi.append(loss.log10().item())
    if i % 500 == 0:
        print(f'{i:5d}/{max_steps}: {loss.item():.4f}')

print('训练结束,最后一步 loss =', round(loss.item(), 4))
plt.plot(lossi); plt.title('手写 backprop 训练的 loss(log10)');

---

## 小结

- **反向传播 = 链式法则 + 一张小导数表 + 形状规则(sum↔broadcast)**,没有更玄的东西。
- **Exercise 1**:把每一步拆开手推,`exact: True` 逐个验证——这是理解「梯度到底怎么流」的最好方式。
- **Exercise 2/3**:softmax+交叉熵、BatchNorm 合并求导得到的紧凑公式,正是框架内部又快又稳的原因。
- **Exercise 4**:证明手写 backprop 真能训练网络。

> 想看这些手推梯度在**工程上怎么用**(梯度检查、数值稳定、梯度裁剪等),打开配套的 `Backprop_Ninja_生产案例.ipynb`——那是全新写的、安全可跑的代码。